In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("patelris/crop-yield-prediction-dataset")
import os

print("Path to dataset files:", path)

c:\Users\Straw\Desktop\Crop Yield Prediction\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\Straw\.cache\kagglehub\datasets\patelris\crop-yield-prediction-dataset\versions\1


In [4]:
files = os.listdir(path)
print("Files:", files)

Files: ['pesticides.csv', 'rainfall.csv', 'temp.csv', 'yield.csv', 'yield_df.csv']


In [2]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv(os.path.join(path, files[4])) 

In [6]:
df.head()

,Unnamed: 0,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,1,Albania,Potatoes,1990,66667,1485.0,121.0,16.37
2,2,Albania,"Rice, paddy",1990,23333,1485.0,121.0,16.37
3,3,Albania,Sorghum,1990,12500,1485.0,121.0,16.37
4,4,Albania,Soybeans,1990,7000,1485.0,121.0,16.37


In [7]:
df = df.drop(columns=['Unnamed: 0'])

In [8]:
df.isnull().sum()

Area                             0
Item                             0
Year                             0
hg/ha_yield                      0
average_rain_fall_mm_per_year    0
pesticides_tonnes                0
avg_temp                         0
dtype: int64

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28242 entries, 0 to 28241
Data columns (total 7 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Area                           28242 non-null  object 
 1   Item                           28242 non-null  object 
 2   Year                           28242 non-null  int64  
 3   hg/ha_yield                    28242 non-null  int64  
 4   average_rain_fall_mm_per_year  28242 non-null  float64
 5   pesticides_tonnes              28242 non-null  float64
 6   avg_temp                       28242 non-null  float64
dtypes: float64(3), int64(2), object(2)
memory usage: 1.5+ MB


# Baseline Model

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [11]:
df = pd.get_dummies(df, columns=['Area', 'Item'], drop_first=True)

In [12]:
TARGET = 'hg/ha_yield'
X = df.drop(columns=[TARGET])
y = df[TARGET]
y = np.log1p(df['hg/ha_yield'])

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [15]:
models = {
    "Linear Regression":  LinearRegression(),
    "Random Forest":      RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting":  GradientBoostingRegressor(n_estimators=100, random_state=42),
}

def evaluate(name, model):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f"\n── {name} ──")
    print(f"  MAE:  {mean_absolute_error(y_test, preds):.2f}")
    print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.2f}")
    print(f"  R2:   {r2_score(y_test, preds):.4f}")
    return model, preds

results = {name: evaluate(name, m) for name, m in models.items()}


rf_model = results["Random Forest"][0]
feature_names = df.drop(columns=[TARGET]).columns
feat_imp = pd.Series(rf_model.feature_importances_, index=feature_names)
print("\nTop 10 features:\n", feat_imp.sort_values(ascending=False).head(10))


── Linear Regression ──
  MAE:  0.34
  RMSE: 0.45
  R2:   0.8347

── Random Forest ──
  MAE:  0.06
  RMSE: 0.14
  R2:   0.9833

── Gradient Boosting ──
  MAE:  0.34
  RMSE: 0.46
  R2:   0.8236

Top 10 features:
 Item_Potatoes                    0.280085
Item_Sweet potatoes              0.106402
Item_Soybeans                    0.074882
pesticides_tonnes                0.073933
avg_temp                         0.064897
Item_Rice, paddy                 0.063123
Item_Sorghum                     0.061398
average_rain_fall_mm_per_year    0.059127
Item_Maize                       0.050971
Item_Wheat                       0.030890
dtype: float64


In [79]:
df_raw = pd.read_csv(os.path.join(path, files[4])) 

print(df_raw.groupby('Item')['hg/ha_yield'].mean().sort_values(ascending=False).head(10))

preds = rf_model.predict(X_test)
residuals = pd.DataFrame({'actual': y_test, 'predicted': preds})
residuals['error'] = abs(residuals['actual'] - residuals['predicted'])
print(residuals.sort_values('error', ascending=False).head(10))

Item
Potatoes                199801.549579
Cassava                 150479.466993
Sweet potatoes          119057.793772
Yams                    114140.345927
Plantains and others    106041.320144
Rice, paddy              40730.434770
Maize                    36310.070614
Wheat                    30116.267825
Sorghum                  18635.777229
Soybeans                 16731.092771
Name: hg/ha_yield, dtype: float64
       actual  predicted      error
15666  301646  170017.14  131628.86
9575   223715  347774.67  124059.67
8874   468991  355026.05  113964.95
8483   312986  200912.38  112073.62
2075   255556  146228.96  109327.04
24407  258305  151582.66  106722.34
194    270526  170017.34  100508.66
23720  253007  153827.30   99179.70
21497  100000  196534.80   96534.80
20952  209796  117185.03   92610.97


In [20]:

df = pd.read_csv(os.path.join(path, files[4])) 
df = df.drop(["Year", "Unnamed: 0"], axis=1)

TARGET = 'hg/ha_yield'
crop_models = {}
crop_results = []

for crop, group in df.groupby('Item'):
    if len(group) < 50:  # skip crops with too little data
        print(f"Skipping {crop} — only {len(group)} rows")
        continue

    # One-hot encode Area (per crop, so no Item column needed)
    group = pd.get_dummies(group.drop(columns=['Item']), columns=['Area'], drop_first=True)
    group['pesticides_tonnes'] = np.log1p(group['pesticides_tonnes'])

    X = group.drop(columns=[TARGET])
    y = group[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    mean_yield = y.mean()

    crop_models[crop] = {'model': model, 'scaler': scaler, 'columns': X.columns.tolist()}
    crop_results.append({
        'Crop': crop, 'Rows': len(group),
        'R2': round(r2, 4),
        'MAE': round(mae, 1),
        'RMSE': round(rmse, 1),
        'Mean Yield': round(mean_yield, 1),
        'MAE %': round((mae / mean_yield) * 100, 1)
    })

# ── SUMMARY
results_df = pd.DataFrame(crop_results).sort_values('R2', ascending=False)
print(results_df.to_string(index=False))

                Crop  Rows     R2     MAE    RMSE  Mean Yield  MAE %
             Cassava  2045 0.9619  8364.9 17323.3    150479.5    5.6
               Wheat  3857 0.9565  2187.5  3857.8     30116.3    7.3
         Rice, paddy  3388 0.9438  2513.2  4614.7     40730.4    6.2
            Potatoes  4276 0.9433 12913.1 23059.3    199801.5    6.5
                Yams   847 0.9387  5992.7 13368.1    114140.3    5.3
               Maize  4121 0.9182  3936.0  7750.0     36310.1   10.8
            Soybeans  3223 0.8969  1343.6  2502.4     16731.1    8.0
      Sweet potatoes  2890 0.8929  8243.5 23586.1    119057.8    6.9
             Sorghum  3039 0.8771  2262.2  4909.9     18635.8   12.1
Plantains and others   556 0.7139 15350.8 34754.9    106041.3   14.5


In [26]:
import joblib, json, os

os.makedirs("models", exist_ok=True)

meta = {}
for crop, info in crop_models.items():
    crop_df = df[df['Item'] == crop]
    meta[crop] = {
        'columns': info['columns'],               
        'countries': sorted(crop_df['Area'].unique().tolist()), 
        'ranges': {              
            col: {
                'min': round(float(crop_df[col].min()), 2),
                'max': round(float(crop_df[col].max()), 2)
            }
            for col in ['average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp']
        }, 
        'country_pesticides': (
            crop_df.groupby('Area')['pesticides_tonnes']
            .mean().round(2).to_dict()
        )
    }
    # Save model and scaler
    filename = crop.replace(" ", "_").replace(",", "").replace("/", "")
    joblib.dump(info['model'],  f"models/{filename}_model.pkl")
    joblib.dump(info['scaler'], f"models/{filename}_scaler.pkl")


with open("models/meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Done! Keys in meta:", list(meta['Maize'].keys()))


Done! Keys in meta: ['columns', 'countries', 'ranges', 'country_pesticides']


In [27]:
from sklearn.model_selection import KFold, cross_val_score

TARGET = 'hg/ha_yield'
crop_models = {}
crop_results = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)  # 5-fold cross validation

for crop, group in df.groupby('Item'):
    if len(group) < 50:
        print(f"Skipping {crop} — only {len(group)} rows")
        continue

    group = pd.get_dummies(group.drop(columns=['Item']), columns=['Area'], drop_first=True)
    group['pesticides_tonnes'] = np.log1p(group['pesticides_tonnes'])

    X = group.drop(columns=[TARGET])
    y = group[TARGET]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestRegressor(n_estimators=100, random_state=42)

    cv_r2   = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2')
    cv_mae  = cross_val_score(model, X_scaled, y, cv=kf, scoring='neg_mean_absolute_error')
    cv_rmse = cross_val_score(model, X_scaled, y, cv=kf, scoring='neg_root_mean_squared_error')

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae       = mean_absolute_error(y_test, preds)
    rmse      = np.sqrt(mean_squared_error(y_test, preds))
    r2        = r2_score(y_test, preds)
    mean_yield = y.mean()

    crop_models[crop] = {'model': model, 'scaler': scaler, 'columns': X.columns.tolist()}
    crop_results.append({
        'Crop':         crop,
        'Rows':         len(group),
        'R2':           round(r2, 4),
        'MAE':          round(mae, 1),
        'RMSE':         round(rmse, 1),
        'Mean Yield':   round(mean_yield, 1),
        'MAE %':        round((mae / mean_yield) * 100, 1),
        # K-Fold cross validation results
        'CV R2 Mean':   round(cv_r2.mean(), 4),
        'CV R2 Std':    round(cv_r2.std(), 4),
        'CV MAE Mean':  round(-cv_mae.mean(), 1),
        'CV RMSE Mean': round(-cv_rmse.mean(), 1),
    })

results_df = pd.DataFrame(crop_results).sort_values('R2', ascending=False)
print(results_df.to_string(index=False))

                Crop  Rows     R2     MAE    RMSE  Mean Yield  MAE %  CV R2 Mean  CV R2 Std  CV MAE Mean  CV RMSE Mean
             Cassava  2045 0.9615  8352.9 17420.3    150479.5    5.6      0.9545     0.0128       9001.0       18795.4
               Wheat  3857 0.9565  2187.0  3857.6     30116.3    7.3      0.9614     0.0054       2025.3        3598.5
         Rice, paddy  3388 0.9438  2514.0  4614.4     40730.4    6.2      0.9340     0.0135       2734.2        4884.3
            Potatoes  4276 0.9433 12926.8 23051.3    199801.5    6.5      0.9406     0.0044      12713.6       22726.9
                Yams   847 0.9387  5994.8 13362.0    114140.3    5.3      0.9392     0.0142       6552.1       13267.3
               Maize  4121 0.9180  3944.0  7758.4     36310.1   10.9      0.9289     0.0097       3675.1        7282.5
            Soybeans  3223 0.8970  1342.8  2500.8     16731.1    8.0      0.9008     0.0085       1299.7        2391.5
      Sweet potatoes  2890 0.8932  8188.4 23556.